In [40]:
import numpy as np
import pandas as pd

from SymbolicDSGE import DSGESolver, ModelParser, Shock
from SymbolicDSGE.monte_carlo import (
    MCPipeline,
    reference_filter_step,
    simulation_step,
    wald_test_step,
    regression_step,
)

In [41]:
model, kalman = ModelParser("../../MODELS/misspec_test/reference.yaml").get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(
    n_state=3,
    n_exog=3,
)

steady_state = np.zeros(5, dtype=np.float64)
reference = solver.solve(compiled, steady_state=steady_state)

dgp_model, dgp_kalman = ModelParser("../../MODELS/misspec_test/misspec.yaml").get_all()
dgp_comp = DSGESolver(dgp_model, dgp_kalman).compile(n_state=3, n_exog=3)
dgp_sol = DSGESolver(dgp_model, dgp_kalman)
dgp = dgp_sol.solve(dgp_comp, steady_state=steady_state)

In [42]:
T = 200
n_obs = len(reference.compiled.observable_names)

pipeline = MCPipeline(
    [
        simulation_step(
            T=T,
            shocks={
                "g,z": Shock(T=T, dist="norm", multivar=True, seed=0),
                "r": Shock(T=T, dist="norm", seed=1),
            },
            observables=True,
        ),
        reference_filter_step(estimate_R_diag=False),
        wald_test_step(
            "std_innov_mean",
            source="std_innov",
            target=np.zeros(n_obs),
            kind="mean",
            burn_in=20,
        ),
        wald_test_step(
            "std_innov_second_moment",
            source="std_innov",
            target=np.eye(n_obs),
            kind="second_moment",
            burn_in=20,
        ),
        regression_step(
            "OutGap_regression",
            X_source="x_pred",
            y_source="y_pred",
            y_columns=[0],
            X_columns=[2, 3, 4],
            variables=["r", "x", "Pi"],
        ),
    ]
)

In [43]:
mc = pipeline.run(
    reference=reference,
    dgp=dgp,
    n_rep=100_000,
    retain_payloads=False,
    retain_test_results=False,
)

mc.succeeded, mc.n_successful

(True, 100000)

In [44]:
t_summary = pd.DataFrame(
    {
        name: {
            "mean_statistic": res.mean_statistic,
            "mean_pval": res.mean_pval,
            "rejection_rate": res.rejection_rate,
            "ci_low": res.pval_confidence_interval()[0],
            "ci_high": res.pval_confidence_interval()[1],
        }
        for name, res in mc.test_summaries.items()
    }
).T
t_summary.round(3)

,mean_statistic,mean_pval,rejection_rate,ci_low,ci_high
std_innov_mean,2.739,0.55,0.051,0.05,0.052
std_innov_second_moment,722.509,0.00,1.000,1.00,1.000


In [45]:
mc.regression_summaries["OutGap_regression"].t_stat_trace

C:\Users\guney\Documents\GitHub\SymbolicDSGE\SymbolicDSGE\regression\ols\ols_result.py:241: RuntimeWarning: divide by zero encountered in divide
  return np.asarray(self.coef_trace / self.se_trace, dtype=np.float64)
C:\Users\guney\Documents\GitHub\SymbolicDSGE\SymbolicDSGE\regression\ols\ols_result.py:241: RuntimeWarning: invalid value encountered in divide
  return np.asarray(self.coef_trace / self.se_trace, dtype=np.float64)


array([[ 1.18126313e+01,  1.08308432e+16, -3.83714486e+00],
       [-9.02108236e+00,  1.20414017e+16, -8.66127074e+00],
       [ 6.23597534e+00,  9.79023680e+15,  1.19236384e+01],
       ...,
       [-1.26734612e+01,  7.59721111e+15,  2.64252436e+00],
       [ 4.03000697e+00,  3.33538808e+16,  1.02339460e+01],
       [-1.07428930e+01,  2.17203879e+16,  1.03783222e+01]],
      shape=(100000, 3))